# Data Cleaning
Loads the datasets and cleans/makes them useable before saving them to the pre-processed folder.

In [1]:
import glob
import os
import json
import ijson
import sys
import csv
sys.path.append("..")  # must come before utils import

import pandas as pd
from collections import defaultdict

from utils.cleaning import (
    drop_irrelevant_columns,
    extract_tweet_text,
    derive_is_retweet_from_text,
    derive_is_reply_from_reply_id,
    derive_v2_flags_from_referenced_tweets,
)


### Cresci-15

In [2]:
RAW        = "../data/raw/cresci-2015"
OUT_USERS  = "../data/pre-processed/cresci_2015/users_cresci_2015.csv"
OUT_TWEETS = "../data/pre-processed/cresci_2015/tweets_cresci_2015.csv"

SUBSETS = {
    "TFP": "human",
    "TWT": "human",
    "FSF": "bot",
    "INT": "bot",
    "E13": "bot",
}

print("Paths set. Subsets to load:", list(SUBSETS.keys()))

Paths set. Subsets to load: ['TFP', 'TWT', 'FSF', 'INT', 'E13']


In [3]:
# get the number of users and load them. Find basic data shape

user_frames = []

for subset, label in SUBSETS.items():
    path = f"{RAW}/{subset}.csv/users.csv"
    df = pd.read_csv(path)
    df["label"]  = label
    df["subset"] = subset
    user_frames.append(df)
    print(f"  {subset}: {len(df)} users ({label})")

users = pd.concat(user_frames, ignore_index=True)
print(f"\nTotal users loaded: {len(users)}")
print(f"Label distribution:\n{users['label'].value_counts().to_string()}")

  TFP: 469 users (human)
  TWT: 845 users (human)
  FSF: 1169 users (bot)
  INT: 1337 users (bot)
  E13: 1481 users (bot)

Total users loaded: 5301
Label distribution:
label
bot      3987
human    1314


In [4]:
# Keep raw columns needed for feature engineering later
KEEP = [
    "id", "screen_name", "label", "subset",
    "followers_count", "friends_count", "statuses_count",
    "listed_count", "created_at",
    "verified", "description", "location", "default_profile_image",
]

users = drop_irrelevant_columns(users, KEEP)
users = users.rename(columns={"id": "user_id"})

Kept 13 columns, dropped 23
Dropped: ['name', 'favourites_count', 'url', 'lang', 'time_zone', 'default_profile', 'geo_enabled', 'profile_image_url', 'profile_banner_url', 'profile_use_background_image', 'profile_background_image_url_https', 'profile_text_color', 'profile_image_url_https', 'profile_sidebar_border_color', 'profile_background_tile', 'profile_sidebar_fill_color', 'profile_background_image_url', 'profile_background_color', 'profile_link_color', 'utc_offset', 'protected', 'updated', 'dataset']
Remaining nulls:
verified                 5301
description              1216
location                 1212
default_profile_image    5276


In [5]:
# Load raw tweet text for all subsets
tweet_frames = []

for subset in SUBSETS:
    path = f"{RAW}/{subset}.csv/tweets.csv"
    df = pd.read_csv(path, low_memory=False, encoding="latin-1")
    tweet_frames.append(df)
    print(f"  {subset}: {len(df)} tweets")

raw_tweets = pd.concat(tweet_frames, ignore_index=True)
print(f"\nTotal tweets loaded: {len(raw_tweets):,}")

tweets = extract_tweet_text(
    raw_tweets, users["user_id"],
    extra_cols=["in_reply_to_user_id"],
)


  TFP: 563693 tweets
  TWT: 114192 tweets
  FSF: 22910 tweets
  INT: 58925 tweets
  E13: 2068037 tweets

Total tweets loaded: 2,827,757
Tweets from labeled users: 2,827,757 / 2,827,757 total


In [6]:
# Save the files
os.makedirs(os.path.dirname(OUT_USERS), exist_ok=True)
users["source"] = "cresci_2015"
users.to_csv(OUT_USERS, index=False)
tweets.to_csv(OUT_TWEETS, index=False)

print(f"Users  → {OUT_USERS}  {users.shape}")
print(f"Tweets → {OUT_TWEETS}  {tweets.shape}")
print(f"User columns:  {list(users.columns)}")
print(f"Tweet columns: {list(tweets.columns)}")
print(f"Label distribution: {users["label"].value_counts().to_string()}")


Users  → ../data/pre-processed/cresci_2015/users_cresci_2015.csv  (5301, 14)
Tweets → ../data/pre-processed/cresci_2015/tweets_cresci_2015.csv  (2827757, 7)
User columns:  ['user_id', 'screen_name', 'label', 'subset', 'followers_count', 'friends_count', 'statuses_count', 'listed_count', 'created_at', 'verified', 'description', 'location', 'default_profile_image', 'source']
Tweet columns: ['tweet_id', 'user_id', 'text', 'created_at', 'in_reply_to_user_id', 'is_retweet', 'is_reply']
Label distribution: label
bot      3987
human    1314


### Cresci-17

In [7]:
RAW_17        = "../data/raw/cresci-2017"
OUT_USERS_17  = "../data/pre-processed/cresci_2017/users_cresci_2017.csv"
OUT_TWEETS_17 = "../data/pre-processed/cresci_2017/tweets_cresci_2017.csv"

SUBSETS_17 = {
    "genuine_accounts.csv":       "human",
    "fake_followers.csv":         "bot",
    "social_spambots_1.csv":      "bot",
    "social_spambots_2.csv":      "bot",
    "social_spambots_3.csv":      "bot",
    "traditional_spambots_1.csv": "bot",
    "traditional_spambots_2.csv": "bot",
    "traditional_spambots_3.csv": "bot",
    "traditional_spambots_4.csv": "bot",
}

NO_TWEETS_17 = {"traditional_spambots_2.csv", "traditional_spambots_3.csv", "traditional_spambots_4.csv"}

print("Paths set. Subsets to load:", list(SUBSETS_17.keys()))

Paths set. Subsets to load: ['genuine_accounts.csv', 'fake_followers.csv', 'social_spambots_1.csv', 'social_spambots_2.csv', 'social_spambots_3.csv', 'traditional_spambots_1.csv', 'traditional_spambots_2.csv', 'traditional_spambots_3.csv', 'traditional_spambots_4.csv']


In [8]:
user_frames_17 = []

for subset, label in SUBSETS_17.items():
    path = f"{RAW_17}/{subset}/users.csv"
    df = pd.read_csv(path, encoding="latin-1")
    df["label"]  = label
    df["subset"] = subset
    user_frames_17.append(df)
    print(f"  {subset}: {len(df)} users ({label})")

users_17 = pd.concat(user_frames_17, ignore_index=True)
users_17["user_id"] = users_17["id"].astype(str)
print(f"\nTotal users: {len(users_17)}")
print(f"user_id sample: {users_17['user_id'].head(3).tolist()}")

  genuine_accounts.csv: 3474 users (human)
  fake_followers.csv: 3351 users (bot)
  social_spambots_1.csv: 991 users (bot)
  social_spambots_2.csv: 3457 users (bot)
  social_spambots_3.csv: 464 users (bot)
  traditional_spambots_1.csv: 1000 users (bot)
  traditional_spambots_2.csv: 100 users (bot)
  traditional_spambots_3.csv: 403 users (bot)
  traditional_spambots_4.csv: 1128 users (bot)

Total users: 14368
user_id sample: ['1502026416', '2492782375', '293212315']


In [9]:
KEEP_17 = [
    "user_id", "screen_name", "label", "subset",
    "followers_count", "friends_count", "statuses_count",
    "listed_count", "created_at",
    "verified", "description", "location", "default_profile_image",
]

users_17 = drop_irrelevant_columns(users_17, KEEP_17)

# Normalize L-suffix Unix-ms timestamps (e.g. "1183552203000L" → ISO string)
# These are old API v1 artefacts — Python 2 long-integer notation for ms since epoch
mask_l = users_17["created_at"].astype(str).str.match(r'^\d+L$', na=False)
if mask_l.any():
    users_17.loc[mask_l, "created_at"] = (
        pd.to_datetime(
            users_17.loc[mask_l, "created_at"].str.rstrip("L").astype(int),
            unit="ms", utc=True,
        ).astype(str)
    )
    print(f"Normalized {mask_l.sum()} L-suffix timestamps → ISO format")

print(f"user_id dtype: {users_17['user_id'].dtype}")
print(f"user_id sample: {users_17['user_id'].head(3).tolist()}")

Kept 13 columns, dropped 32
Dropped: ['id', 'name', 'favourites_count', 'url', 'lang', 'time_zone', 'default_profile', 'geo_enabled', 'profile_image_url', 'profile_banner_url', 'profile_use_background_image', 'profile_background_image_url_https', 'profile_text_color', 'profile_image_url_https', 'profile_sidebar_border_color', 'profile_background_tile', 'profile_sidebar_fill_color', 'profile_background_image_url', 'profile_background_color', 'profile_link_color', 'utc_offset', 'is_translator', 'follow_request_sent', 'protected', 'notifications', 'contributors_enabled', 'following', 'timestamp', 'crawled_at', 'updated', 'test_set_1', 'test_set_2']
Remaining nulls:
verified                 14357
description               5744
location                  6684
default_profile_image    14290
Normalized 1000 L-suffix timestamps → ISO format
user_id dtype: str
user_id sample: ['1502026416', '2492782375', '293212315']


In [10]:
tweet_frames_17 = []

for subset in SUBSETS_17:
    if subset in NO_TWEETS_17:
        print(f"  {subset}: no tweets — skipping")
        continue
    df = pd.read_csv(f"{RAW_17}/{subset}/tweets.csv",
                     low_memory=False, encoding="latin-1")
    tweet_frames_17.append(df)
    print(f"  {subset}: {len(df)} tweets")

raw_tweets_17 = pd.concat(tweet_frames_17, ignore_index=True)

# float64 upcast from NaN rows — normalize before matching
raw_tweets_17 = raw_tweets_17.dropna(subset=["user_id"])
raw_tweets_17["user_id"] = raw_tweets_17["user_id"].astype(float).astype(int).astype(str)

print(f"\nTotal tweets: {len(raw_tweets_17):,}")
print(f"tweet user_id sample: {raw_tweets_17['user_id'].head(3).tolist()}")

# filter inline — no utility function
labeled_ids_17 = set(users_17["user_id"])
mask = raw_tweets_17["user_id"].isin(labeled_ids_17)
print(f"Matched: {mask.sum():,} / {len(raw_tweets_17):,}")

tweets_17 = (
    raw_tweets_17[mask][["id", "user_id", "text", "created_at", "in_reply_to_user_id"]]
    .rename(columns={"id": "tweet_id"})
    .copy()
)
tweets_17["tweet_id"] = tweets_17["tweet_id"].astype(str)

# Normalize reply IDs: stored as float due to NaN — convert to int string
tweets_17["in_reply_to_user_id"] = tweets_17["in_reply_to_user_id"].apply(
    lambda x: str(int(x)) if pd.notna(x) else None
)

# Canonical retweet/reply flags — derived once at cleaning so downstream FE
# never has to re-do per-API-version detection.
tweets_17["is_retweet"] = derive_is_retweet_from_text(tweets_17["text"])
tweets_17["is_reply"]   = derive_is_reply_from_reply_id(tweets_17["in_reply_to_user_id"])

print(f"tweets_17 shape: {tweets_17.shape}")
print(f"is_retweet rate: {tweets_17['is_retweet'].mean():.3f}")
print(f"is_reply rate:   {tweets_17['is_reply'].mean():.3f}")


  genuine_accounts.csv: 2839362 tweets
  fake_followers.csv: 196027 tweets
  social_spambots_1.csv: 1610034 tweets
  social_spambots_2.csv: 428542 tweets
  social_spambots_3.csv: 1418557 tweets
  traditional_spambots_1.csv: 145094 tweets
  traditional_spambots_2.csv: no tweets — skipping
  traditional_spambots_3.csv: no tweets — skipping
  traditional_spambots_4.csv: no tweets — skipping

Total tweets: 6,637,615
tweet user_id sample: ['678033', '678033', '678033']
Matched: 6,637,615 / 6,637,615
tweets_17 shape: (6637615, 7)
is_retweet rate: 0.126
is_reply rate:   0.157


In [11]:
# save
os.makedirs(os.path.dirname(OUT_USERS_17), exist_ok=True)
users_17["source"] = "cresci_2017"
users_17.to_csv(OUT_USERS_17, index=False)
tweets_17.to_csv(OUT_TWEETS_17, index=False)

print(f"Users  → {OUT_USERS_17}  {users_17.shape}")
print(f"Tweets → {OUT_TWEETS_17}  {tweets_17.shape}")
print(f"User columns:  {list(users_17.columns)}")
print(f"Tweet columns: {list(tweets_17.columns)}")
print(f"Label distribution: {users_17["label"].value_counts().to_string()}")


Users  → ../data/pre-processed/cresci_2017/users_cresci_2017.csv  (14368, 14)
Tweets → ../data/pre-processed/cresci_2017/tweets_cresci_2017.csv  (6637615, 7)
User columns:  ['user_id', 'screen_name', 'label', 'subset', 'followers_count', 'friends_count', 'statuses_count', 'listed_count', 'created_at', 'verified', 'description', 'location', 'default_profile_image', 'source']
Tweet columns: ['tweet_id', 'user_id', 'text', 'created_at', 'in_reply_to_user_id', 'is_retweet', 'is_reply']
Label distribution: label
bot      10894
human     3474


### Twibot-20

In [12]:
RAW_20        = "../data/raw/Twibot20"
OUT_USERS_20  = "../data/pre-processed/twibot_2020/users_twibot_20.csv"
OUT_TWEETS_20 = "../data/pre-processed/twibot_2020/tweets_twibot_20.csv"

SPLITS_20 = ["train", "test", "dev"]
LABEL_MAP = {"0": "human", "1": "bot"}

print("Paths set. Splits to load:", SPLITS_20)

Paths set. Splits to load: ['train', 'test', 'dev']


In [13]:
# loading records
user_records  = []
tweet_records = []

for split in SPLITS_20:
    with open(f"{RAW_20}/{split}.json") as f:
        data = json.load(f)

    for rec in data:
        uid   = str(rec["ID"])
        label = LABEL_MAP[str(rec["label"])]

        user_records.append({**rec["profile"], "user_id": uid, "label": label, "subset": split})

        if rec["tweet"] is not None:
            for text in rec["tweet"]:
                tweet_records.append({
                    "tweet_id":            None,
                    "user_id":             uid,
                    "text":                text,
                    "created_at":          None,
                    "in_reply_to_user_id": None,
                })

    print(f"  {split}: {len(data)} users")

print(f"Total users:  {len(user_records):,}")
print(f"Total tweets: {len(tweet_records):,}")


  train: 8278 users
  test: 1183 users
  dev: 2365 users
Total users:  11,826
Total tweets: 1,999,788


In [14]:
# drop columns
users_20 = pd.DataFrame(user_records)

# profile already has v1 flat fields — rename id to user_id if it exists as a separate column
if "id" in users_20.columns:
    users_20 = users_20.drop(columns=["id"])

print(f"Available profile columns: {list(users_20.columns)}")
print(f"\nLabel distribution:\n{users_20['label'].value_counts().to_string()}")
print(f"Split distribution:\n{users_20['subset'].value_counts().to_string()}")

KEEP_20 = [
    "user_id", "screen_name", "label", "subset",
    "followers_count", "friends_count", "statuses_count",
    "listed_count", "created_at",
    "verified", "description", "location", "default_profile_image",
]

users_20 = drop_irrelevant_columns(users_20, KEEP_20)

Available profile columns: ['id_str', 'name', 'screen_name', 'location', 'profile_location', 'description', 'url', 'entities', 'protected', 'followers_count', 'friends_count', 'listed_count', 'created_at', 'favourites_count', 'utc_offset', 'time_zone', 'geo_enabled', 'verified', 'statuses_count', 'lang', 'contributors_enabled', 'is_translator', 'is_translation_enabled', 'profile_background_color', 'profile_background_image_url', 'profile_background_image_url_https', 'profile_background_tile', 'profile_image_url', 'profile_image_url_https', 'profile_link_color', 'profile_sidebar_border_color', 'profile_sidebar_fill_color', 'profile_text_color', 'profile_use_background_image', 'has_extended_profile', 'default_profile', 'default_profile_image', 'user_id', 'label', 'subset']

Label distribution:
label
bot      6589
human    5237
Split distribution:
subset
train    8278
dev      2365
test     1183
Kept 13 columns, dropped 27
Dropped: ['id_str', 'name', 'profile_location', 'url', 'entities',

In [15]:
# tweets
tweets_20 = pd.DataFrame(tweet_records)
tweets_20["user_id"] = tweets_20["user_id"].astype(str)

# Filter to only labeled users (exclude any neighbor bleed-through)
tweets_20 = tweets_20[tweets_20["user_id"].isin(users_20["user_id"].astype(str))]

# is_retweet derivable from "RT @" text prefix; is_reply unrecoverable from
# this dataset (the original format dropped in_reply_to_user_id) — set to
# pd.NA so FE treats it as missing rather than 0.
tweets_20["is_retweet"] = derive_is_retweet_from_text(tweets_20["text"])
tweets_20["is_reply"]   = pd.NA

print(f"Tweet rows: {len(tweets_20):,}")
print(f"Unique users with tweets: {tweets_20['user_id'].nunique():,}")
print(f"Columns: {list(tweets_20.columns)}")
print(f"is_retweet rate: {tweets_20['is_retweet'].mean():.3f}")
print("Note: no tweet_id, created_at, or in_reply_to_user_id — original format")
print("      stored tweets as plain strings; reply detection unrecoverable.")


Tweet rows: 1,999,788
Unique users with tweets: 11,746
Columns: ['tweet_id', 'user_id', 'text', 'created_at', 'in_reply_to_user_id', 'is_retweet', 'is_reply']
is_retweet rate: 0.322
Note: no tweet_id, created_at, or in_reply_to_user_id — original format
      stored tweets as plain strings; reply detection unrecoverable.


In [16]:
# save
os.makedirs(os.path.dirname(OUT_USERS_20), exist_ok=True)
users_20["source"] = "twibot_20"
users_20.to_csv(OUT_USERS_20, index=False)
tweets_20.to_csv(OUT_TWEETS_20, index=False)

print(f"Users  → {OUT_USERS_20}  {users_20.shape}")
print(f"Tweets → {OUT_TWEETS_20}  {tweets_20.shape}")
print(f"User columns:  {list(users_20.columns)}")
print(f"Tweet columns: {list(tweets_20.columns)}")
print(f"Label distribution:{users_20["label"].value_counts().to_string()}")


Users  → ../data/pre-processed/twibot_2020/users_twibot_20.csv  (11826, 14)
Tweets → ../data/pre-processed/twibot_2020/tweets_twibot_20.csv  (1999788, 7)
User columns:  ['user_id', 'screen_name', 'label', 'subset', 'followers_count', 'friends_count', 'statuses_count', 'listed_count', 'created_at', 'verified', 'description', 'location', 'default_profile_image', 'source']
Tweet columns: ['tweet_id', 'user_id', 'text', 'created_at', 'in_reply_to_user_id', 'is_retweet', 'is_reply']
Label distribution:label
bot      6589
human    5237


### Twibot-22

In [17]:
RAW_22        = "../data/raw/Twibot22"
OUT_USERS_22  = "../data/pre-processed/twibot_2022/users_twibot_22.csv"
OUT_TWEETS_22 = "../data/pre-processed/twibot_2022/tweets_twibot_22.csv"

print("Paths set.")

Paths set.


In [18]:
# Load labels, splits, and users
labels_22 = pd.read_csv(f"{RAW_22}/label.csv")
labels_22.columns = ["user_id", "label"]
labels_22["user_id"] = labels_22["user_id"].astype(str)

splits_22 = pd.read_csv(f"{RAW_22}/split.csv")
splits_22.columns = ["user_id", "subset"]
splits_22["user_id"] = splits_22["user_id"].astype(str)

labeled_ids = set(labels_22["user_id"])
print(f"Labeled users: {len(labeled_ids):,}")
print(f"Label distribution:\n{labels_22['label'].value_counts().to_string()}")
print(f"\nSplit distribution:\n{splits_22['subset'].value_counts().to_string()}")

print("\nLoading user.json...")
with open(f"{RAW_22}/user.json") as f:
    raw_users = json.load(f)
print(f"Total users in user.json: {len(raw_users):,}")

Labeled users: 1,000,000
Label distribution:
label
human    860057
bot      139943

Split distribution:
subset
train    700000
val      200000
test     100000

Loading user.json...
Total users in user.json: 1,000,000


In [19]:
# build dataframe
user_records = []

for u in raw_users:
    uid = str(u["id"])
    if uid not in labeled_ids:
        continue

    pm = u.get("public_metrics") or {}
    user_records.append({
        "user_id":               uid,
        "screen_name":           u.get("username"),
        "followers_count":       pm.get("followers_count"),
        "friends_count":         pm.get("following_count"),
        "statuses_count":        pm.get("tweet_count"),
        "listed_count":          pm.get("listed_count"),
        "created_at":            u.get("created_at"),
        "verified":              u.get("verified"),
        "description":           u.get("description"),
        "location":              u.get("location"),
        "default_profile_image": int("default_profile" in str(u.get("profile_image_url", ""))),
    })

users_22 = pd.DataFrame(user_records)
users_22 = users_22.merge(labels_22, on="user_id", how="left")
users_22 = users_22.merge(splits_22, on="user_id", how="left")

# reorder to match the other three datasets
COL_ORDER = [
    "user_id", "screen_name", "label", "subset",
    "followers_count", "friends_count", "statuses_count",
    "listed_count", "created_at",
    "verified", "description", "location", "default_profile_image",
]
users_22 = users_22[COL_ORDER]

print(f"Labeled users extracted: {len(users_22):,}")
print(f"Remaining nulls: {users_22.isnull().sum()[users_22.isnull().sum() > 0].to_string()}")

Labeled users extracted: 1,000,000
Remaining nulls: location    291542


In [20]:
# Stream tweets directly to CSV — never accumulates in RAM
tweet_shards = sorted(glob.glob(f"{RAW_22}/tweet_*.json"))
print(f"Tweet shards found: {len(tweet_shards)}")

os.makedirs(os.path.dirname(OUT_TWEETS_22), exist_ok=True)

total_collected = 0
total_retweets  = 0
total_replies   = 0

with open(OUT_TWEETS_22, "w", encoding="utf-8", newline="") as csv_out:
    writer = csv.DictWriter(
        csv_out,
        fieldnames=[
            "tweet_id", "user_id", "text", "created_at",
            "in_reply_to_user_id", "is_retweet", "is_reply",
        ],
    )
    writer.writeheader()

    for shard_path in tweet_shards:
        shard_name      = os.path.basename(shard_path)
        shard_collected = 0

        with open(shard_path, "rb") as f:
            for rec in ijson.items(f, "item"):
                uid = "u" + str(rec["author_id"])
                if uid not in labeled_ids:
                    continue
                reply_uid = rec.get("in_reply_to_user_id")

                # v2 canonical retweet/reply markers from referenced_tweets.
                # If the field isn't present the helper returns (0, 0).
                v2_rt, v2_rep = derive_v2_flags_from_referenced_tweets(
                    rec.get("referenced_tweets")
                )
                # Reply also true if in_reply_to_user_id is set (defensive).
                is_reply = int(v2_rep or bool(reply_uid))
                is_retweet = v2_rt

                writer.writerow({
                    "tweet_id":            str(rec["id"]),
                    "user_id":             uid,
                    "text":                rec.get("text", ""),
                    "created_at":          rec.get("created_at", ""),
                    "in_reply_to_user_id": ("u" + str(reply_uid)) if reply_uid else "",
                    "is_retweet":          is_retweet,
                    "is_reply":            is_reply,
                })
                shard_collected += 1
                total_collected += 1
                total_retweets  += is_retweet
                total_replies   += is_reply

        print(f"  {shard_name}: {shard_collected:,} tweets  "
              f"(running total: {total_collected:,})")

print(f"\nTotal tweets written: {total_collected:,}")
if total_collected:
    print(f"is_retweet rate: {total_retweets / total_collected:.3f}")
    print(f"is_reply rate:   {total_replies  / total_collected:.3f}")


Tweet shards found: 9
  tweet_0-006.json: 10,000,000 tweets  (running total: 10,000,000)
  tweet_1-005.json: 10,000,000 tweets  (running total: 20,000,000)
  tweet_2-009.json: 10,000,000 tweets  (running total: 30,000,000)
  tweet_3-008.json: 10,000,000 tweets  (running total: 40,000,000)
  tweet_4-001.json: 10,000,000 tweets  (running total: 50,000,000)
  tweet_5-004.json: 10,000,000 tweets  (running total: 60,000,000)
  tweet_6-011.json: 10,000,000 tweets  (running total: 70,000,000)
  tweet_7-010.json: 10,000,000 tweets  (running total: 80,000,000)
  tweet_8-007.json: 8,217,457 tweets  (running total: 88,217,457)

Total tweets written: 88,217,457
is_retweet rate: 0.027
is_reply rate:   0.280


In [21]:
# tweets already written to disk in previous cell
os.makedirs(os.path.dirname(OUT_USERS_22), exist_ok=True)
users_22["source"] = "twibot_22"
users_22.to_csv(OUT_USERS_22, index=False)

print(f"Users  → {OUT_USERS_22}  {users_22.shape}")
print(f"Tweets → {OUT_TWEETS_22}  ({total_collected:,} rows)")
print(f"User columns:  {list(users_22.columns)}")
print(f"Label distribution:{users_22["label"].value_counts().to_string()}")


Users  → ../data/pre-processed/twibot_2022/users_twibot_22.csv  (1000000, 14)
Tweets → ../data/pre-processed/twibot_2022/tweets_twibot_22.csv  (88,217,457 rows)
User columns:  ['user_id', 'screen_name', 'label', 'subset', 'followers_count', 'friends_count', 'statuses_count', 'listed_count', 'created_at', 'verified', 'description', 'location', 'default_profile_image', 'source']
Label distribution:label
human    860057
bot      139943


### Verification

In [22]:
preprocessed_files = [
    "../data/pre-processed/cresci_2015/users_cresci_2015.csv",
    "../data/pre-processed/cresci_2015/tweets_cresci_2015.csv",
    "../data/pre-processed/cresci_2017/users_cresci_2017.csv",
    "../data/pre-processed/cresci_2017/tweets_cresci_2017.csv",
    "../data/pre-processed/twibot_2020/users_twibot_20.csv",
    "../data/pre-processed/twibot_2020/tweets_twibot_20.csv",
    "../data/pre-processed/twibot_2022/users_twibot_22.csv",
    "../data/pre-processed/twibot_2022/tweets_twibot_22.csv",
]

for path in preprocessed_files:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        # use csv.reader to count actual rows (handles embedded newlines in text fields)
        with open(path, encoding="utf-8", newline="") as fh:
            rows = sum(1 for _ in csv.reader(fh)) - 1  # subtract header
        print(f"{os.path.basename(path):<35} {size_mb:>8.1f} MB   {rows:>12,} rows")
    else:
        print(f"{os.path.basename(path):<35} NOT FOUND")


users_cresci_2015.csv                    0.9 MB          5,301 rows
tweets_cresci_2015.csv                 450.7 MB      2,827,757 rows
users_cresci_2017.csv                    2.4 MB         14,368 rows
tweets_cresci_2017.csv                 989.8 MB      6,637,615 rows
users_twibot_20.csv                      2.5 MB         11,826 rows
tweets_twibot_20.csv                   287.9 MB      1,999,788 rows
users_twibot_22.csv                    207.0 MB      1,000,000 rows
tweets_twibot_22.csv                 18650.1 MB     88,217,457 rows


In [23]:
datasets = {
    "cresci_2015": ("../data/pre-processed/cresci_2015/users_cresci_2015.csv",
                    "../data/pre-processed/cresci_2015/tweets_cresci_2015.csv"),
    "cresci_2017": ("../data/pre-processed/cresci_2017/users_cresci_2017.csv",
                    "../data/pre-processed/cresci_2017/tweets_cresci_2017.csv"),
    "twibot_2020": ("../data/pre-processed/twibot_2020/users_twibot_20.csv",
                    "../data/pre-processed/twibot_2020/tweets_twibot_20.csv"),
    "twibot_2022": ("../data/pre-processed/twibot_2022/users_twibot_22.csv",
                    "../data/pre-processed/twibot_2022/tweets_twibot_22.csv"),
}

for name, (users_path, tweets_path) in datasets.items():
    u_cols = list(pd.read_csv(users_path, nrows=0).columns)
    t_cols = list(pd.read_csv(tweets_path, nrows=0).columns)
    print(f"{name}")
    print(f"  users  ({len(u_cols)} cols): {u_cols}")
    print(f"  tweets ({len(t_cols)} cols): {t_cols}")
    print()


cresci_2015
  users  (14 cols): ['user_id', 'screen_name', 'label', 'subset', 'followers_count', 'friends_count', 'statuses_count', 'listed_count', 'created_at', 'verified', 'description', 'location', 'default_profile_image', 'source']
  tweets (7 cols): ['tweet_id', 'user_id', 'text', 'created_at', 'in_reply_to_user_id', 'is_retweet', 'is_reply']

cresci_2017
  users  (14 cols): ['user_id', 'screen_name', 'label', 'subset', 'followers_count', 'friends_count', 'statuses_count', 'listed_count', 'created_at', 'verified', 'description', 'location', 'default_profile_image', 'source']
  tweets (7 cols): ['tweet_id', 'user_id', 'text', 'created_at', 'in_reply_to_user_id', 'is_retweet', 'is_reply']

twibot_2020
  users  (14 cols): ['user_id', 'screen_name', 'label', 'subset', 'followers_count', 'friends_count', 'statuses_count', 'listed_count', 'created_at', 'verified', 'description', 'location', 'default_profile_image', 'source']
  tweets (7 cols): ['tweet_id', 'user_id', 'text', 'created_at